In [2]:
import os, sys, json, shutil, subprocess

# 1. Mount Google Drive & install dependencies
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio>=6.0.0', 'timm'], check=True)

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import timm
import gradio as gr

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = DEVICE == 'cuda'

# 2. Direct paths in Google Drive (no recursive Drive scanning)
DRIVE_ROOT = '/content/drive/MyDrive/COS30082'
CKPT_PATH = os.path.join(DRIVE_ROOT, 'outputs', 'model5_best.pt')
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"Checkpoint not found at: {CKPT_PATH}")

METRICS_PATH = os.path.join(DRIVE_ROOT, 'outputs', 'model5_final_metrics.json')
if not os.path.exists(METRICS_PATH):
    METRICS_PATH = os.path.join(DRIVE_ROOT, 'outputs', 'model5_factorized_vit', 'model5_final_metrics.json')

CACHE_DIR = '/content/dpd_small_320'
IMAGE_DIR = os.path.join(CACHE_DIR, 'DPD_Small_dataset')

# Unpack dataset cache to Colab local SSD for instant sample loading
if not os.path.exists(os.path.join(CACHE_DIR, '.complete')):
    cache_zip = os.path.join(DRIVE_ROOT, 'cache', 'dpd_small_320.zip')
    if not os.path.exists(cache_zip):
        cache_zip = os.path.join(DRIVE_ROOT, 'dpd_small_320.zip')
    if not os.path.exists(cache_zip):
        raise FileNotFoundError(f"Dataset archive not found at {DRIVE_ROOT}/cache/dpd_small_320.zip")
    shutil.copyfile(cache_zip, '/content/dpd_small_320.zip')
    subprocess.run(['unzip', '-q', '/content/dpd_small_320.zip', '-d', '/content'], check=True)
    os.remove('/content/dpd_small_320.zip')

# 3. Rebuild exact 55-crop, 175-disease, and 333-composition mappings
COLS = ['file', 'crop', 'disease', 'comp', 'crop_name', 'disease_name', 'comp_name']
train_df = pd.read_csv(os.path.join(CACHE_DIR, 'DPD_Small_train.csv'), header=None, names=COLS)
test_seen_df = pd.read_csv(os.path.join(CACHE_DIR, 'DPD_Small_test_seen.csv'), header=None, names=COLS)
test_unseen_df = pd.read_csv(os.path.join(CACHE_DIR, 'DPD_Small_test_unseen.csv'), header=None, names=COLS)
all_df = pd.concat([train_df, test_seen_df, test_unseen_df], ignore_index=True)

comps = (
    all_df.drop_duplicates('comp')
    .sort_values('comp')[['comp', 'crop', 'disease', 'crop_name', 'disease_name', 'comp_name']]
    .reset_index(drop=True)
)

NUM_CROPS = int(all_df['crop'].max()) + 1          # 55
NUM_DISEASES = int(all_df['disease'].max()) + 1    # 175
NUM_COMPS = len(comps)                             # 333

COMP_CROP = torch.tensor(comps['crop'].values)
COMP_DIS = torch.tensor(comps['disease'].values)
COMP_NAMES = comps['comp_name'].tolist()
CROP_NAMES = all_df.drop_duplicates('crop').set_index('crop')['crop_name'].sort_index().tolist()
DISEASE_NAMES = all_df.drop_duplicates('disease').set_index('disease')['disease_name'].sort_index().tolist()

FIT_COMPS = sorted(set(train_df['comp']))
NOVEL_MASK = torch.ones(NUM_COMPS, dtype=torch.bool)
NOVEL_MASK[FIT_COMPS] = False

# 4. Load Compositional Prior Gate calibration parameters
if os.path.exists(METRICS_PATH):
    with open(METRICS_PATH, 'r') as f:
        gp = json.load(f)['gating_parameters']
    B0_ANCHOR = float(gp['b0_anchor'])
    W_GATE = torch.tensor(gp['weights_crop_disease'], dtype=torch.float32)
    MU_E = torch.tensor(gp['entropy_norm_mean'], dtype=torch.float32)
    SIG_E = torch.tensor(gp['entropy_norm_std'], dtype=torch.float32)
    Z_MIN, Z_MAX = gp['z_clip_range']
else:
    B0_ANCHOR = 8.950
    W_GATE = torch.tensor([-0.5, 1.0], dtype=torch.float32)
    MU_E = torch.tensor([0.05, 0.08], dtype=torch.float32)
    SIG_E = torch.tensor([0.05, 0.08], dtype=torch.float32)
    Z_MIN, Z_MAX = -2.0, 2.0

# 5. Define & load Model 5 (Orthogonal Subspace Factorized ViT-B/16)
class Model5FactorizedViT(nn.Module):
    def __init__(self, backbone_name, n_crops, n_diseases, proj_dim=128):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0)
        self.sub_dim = self.backbone.num_features // 2  # 384-D crop + 384-D disease
        self.crop_proj = nn.Sequential(
            nn.Linear(self.sub_dim, self.sub_dim), nn.GELU(), nn.Linear(self.sub_dim, proj_dim)
        )
        self.dis_proj = nn.Sequential(
            nn.Linear(self.sub_dim, self.sub_dim), nn.GELU(), nn.Linear(self.sub_dim, proj_dim)
        )
        self.crop_head = nn.Linear(self.sub_dim, n_crops)
        self.disease_head = nn.Linear(self.sub_dim, n_diseases)

    def forward(self, x):
        feat = self.backbone(x)
        return self.crop_head(feat[:, :self.sub_dim]), self.disease_head(feat[:, self.sub_dim:]), feat

model5 = Model5FactorizedViT('vit_base_patch16_224', NUM_CROPS, NUM_DISEASES).to(DEVICE)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model5.load_state_dict(ckpt['model'])
model5.eval()

eval_tf = transforms.Compose([
    transforms.Resize((224, 224), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def format_label(raw_name):
    return raw_name.replace('_', ' ').title()

# 6. Inference & Subspace Diagnostic Pipeline
@torch.no_grad()
def predict_leaf(image, gating_mode, ground_truth_note=""):
    if image is None:
        return {}, {}, {}, "Upload a leaf image or click one of the test examples below."

    x = eval_tf(image.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
        lc, ld, feat = model5(x)

    lc, ld, feat = lc.float().cpu().squeeze(0), ld.float().cpu().squeeze(0), feat.float().cpu().squeeze(0)

    # Inter-subspace cosine overlap |cos(z_crop, z_disease)|
    zc = F.normalize(feat[:384], dim=-1)
    zd = F.normalize(feat[384:], dim=-1)
    abs_cos = float((zc * zd).sum().abs().item())

    # Subspace probabilities & normalized Shannon entropies
    pc, pd = F.softmax(lc, dim=0), F.softmax(ld, dim=0)
    Hc = float(-(pc * torch.log(pc.clamp(min=1e-12))).sum().item() / np.log(NUM_CROPS))
    Hd = float(-(pd * torch.log(pd.clamp(min=1e-12))).sum().item() / np.log(NUM_DISEASES))

    # Dynamic novelty offset g(x)
    if gating_mode.startswith("Compositional Prior"):
        z_ent = ((torch.tensor([Hc, Hd]) - MU_E) / SIG_E).clamp(Z_MIN, Z_MAX)
        g_x = float(B0_ANCHOR + torch.dot(z_ent, W_GATE).item())
    elif gating_mode.startswith("Static Scalar"):
        g_x = float(B0_ANCHOR)
    else:
        g_x = 0.0

    # Joint calibrated score across all 333 valid pairs
    raw_pair_scores = F.log_softmax(lc, dim=0)[COMP_CROP] + F.log_softmax(ld, dim=0)[COMP_DIS]
    cal_pair_scores = raw_pair_scores + g_x * NOVEL_MASK.float()
    pair_probs = F.softmax(cal_pair_scores, dim=0)

    top5_probs, top5_idx = pair_probs.topk(5)
    comp_dict = {
        f"{'[UNSEEN]' if NOVEL_MASK[idx] else '[SEEN]'} {format_label(COMP_NAMES[idx])}": float(prob)
        for idx, prob in zip(top5_idx.tolist(), top5_probs.tolist())
    }

    c_probs, c_idx = pc.topk(3)
    crop_dict = {format_label(CROP_NAMES[i]): float(p) for i, p in zip(c_idx.tolist(), c_probs.tolist())}

    d_probs, d_idx = pd.topk(3)
    dis_dict = {format_label(DISEASE_NAMES[i]): float(p) for i, p in zip(d_idx.tolist(), d_probs.tolist())}

    # Seen vs. Unseen routing telemetry
    margin = float(cal_pair_scores[NOVEL_MASK].max().item() - cal_pair_scores[~NOVEL_MASK].max().item())
    top1_id = int(top5_idx[0].item())
    top1_status = "UNSEEN (Zero-Shot Novel Pair)" if NOVEL_MASK[top1_id] else "SEEN (Training Pair)"
    gt_line = f"- **Sample Ground Truth:** `{ground_truth_note}`\n" if ground_truth_note else ""

    telemetry = (
        f"### Model 5 Subspace & Routing Telemetry\n"
        f"{gt_line}"
        f"- **Top-1 Prediction:** **{format_label(COMP_NAMES[top1_id])}** — `{top1_status}`\n"
        f"- **Dynamic Novelty Boost $g(x)$:** `{g_x:.3f}` *(Static Anchor $b_0 = {B0_ANCHOR:.3f}$)*\n"
        f"- **Unseen vs. Seen Margin:** `{margin:+.3f}` *(> 0 routes to Unseen)*\n"
        f"- **Normalized Subspace Entropy:** Crop $H_c = {Hc:.4f}$ | Disease $H_d = {Hd:.4f}$\n"
        f"- **Subspace Cosine Overlap $|\\cos(z_c, z_d)|$:** `{abs_cos:.4f}`"
    )
    return comp_dict, crop_dict, dis_dict, telemetry

# 7. Curate clickable Unseen & Seen examples for live demo
example_rows, used_files = [], set()
target_examples = [
    (test_unseen_df, 'tomato_early_blight', '[UNSEEN] tomato_early_blight'),
    (test_unseen_df, 'sugarcane_rust', '[UNSEEN] sugarcane_rust'),
    (test_unseen_df, 'corn_brown_spot', '[UNSEEN] corn_brown_spot'),
    (test_unseen_df, 'cucumber_downy_mildew', '[UNSEEN] cucumber_downy_mildew'),
    (test_seen_df, None, None),
    (test_seen_df, None, None),
]

for df_src, comp_filter, label_note in target_examples:
    sub = df_src[df_src['comp_name'] == comp_filter] if comp_filter else df_src[~df_src['file'].isin(used_files)]
    if len(sub) > 0:
        row = sub.iloc[0]
        img_path = os.path.join(IMAGE_DIR, row['file'])
        if os.path.exists(img_path):
            used_files.add(row['file'])
            note = label_note if label_note else f"[SEEN] {row['comp_name']}"
            example_rows.append([img_path, "Compositional Prior Gate (Recommended, HM=64.72%)", note])

# 8. Build & launch Gradio 6 app
with gr.Blocks(title="COS30082 — Model 5 Compositional Plant Disease Classifier") as demo:
    gr.Markdown(
        "# Compositional Plant Disease Classification (Model 5: Orthogonal Subspace ViT-B/16)\n"
        "Partitions the `768-D` ViT `[CLS]` representation into orthogonal `384-D` **Crop** and **Disease** subspaces "
        "with an entropy-aware **Compositional Prior Gate** (`Calibrated HM = 64.72%`, `Unseen@1 = 67.82%`, `Shortcut Rate = 18.92%`)."
    )

    with gr.Row():
        with gr.Column(scale=1):
            img_in = gr.Image(type="pil", label="Leaf Image Input", height=320)
            mode_in = gr.Radio(
                choices=[
                    "Compositional Prior Gate (Recommended, HM=64.72%)",
                    "Static Scalar Bias (b*=8.95, HM=64.37%)",
                    "Unbiased Multi-Task (b=0.0, HM=10.83%)",
                ],
                value="Compositional Prior Gate (Recommended, HM=64.72%)",
                label="Seen / Unseen Calibration Strategy",
            )
            gt_in = gr.Textbox(label="Sample Ground Truth (Auto-filled by Examples)", interactive=False)
            run_btn = gr.Button("Analyze Leaf Composition", variant="primary")

        with gr.Column(scale=1):
            comp_out = gr.Label(num_top_classes=5, label="Top-5 Composite (Crop + Disease) Predictions")
            with gr.Row():
                crop_out = gr.Label(num_top_classes=3, label="Crop Subspace (z_crop, 384-D)")
                dis_out = gr.Label(num_top_classes=3, label="Disease Subspace (z_disease, 384-D)")
            diag_out = gr.Markdown("### Model 5 Subspace & Routing Telemetry\n*Upload an image or click an example below.*")

    for trigger in (run_btn.click, img_in.change, mode_in.change):
        trigger(
            fn=predict_leaf,
            inputs=[img_in, mode_in, gt_in],
            outputs=[comp_out, crop_out, dis_out, diag_out],
        )

    if example_rows:
        gr.Examples(
            examples=example_rows,
            inputs=[img_in, mode_in, gt_in],
            outputs=[comp_out, crop_out, dis_out, diag_out],
            fn=predict_leaf,
            cache_examples=False,
            label="Click a Test Set Sample (Unseen & Seen Pairs)",
        )

demo.launch(share=True, debug=False, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://581ce0fe0e806e365d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
